# 1단계 1차 모집단 수집

`approved + candidate` 소스를 사용해 첫 공고 스냅샷을 수집하고 `staging`에 적재한다. 이 단계에서는 `master`를 갱신하지 않는다.

In [ ]:
from IPython.display import display
import json
import pandas as pd

from jobs_market_v2.pipelines import build_coverage_report_pipeline, collect_jobs_pipeline, verify_sources_pipeline
from jobs_market_v2.settings import get_paths

# Keep this False for CI/smoke tests. Set True when you intentionally want a live bootstrap collection run.
RUN_LIVE_COLLECTION = False

paths = get_paths()
if RUN_LIVE_COLLECTION:
    verify_summary = verify_sources_pipeline()
    collect_summary = collect_jobs_pipeline(dry_run=False)
else:
    verify_summary = {"mode": "smoke", "skipped": "verify_sources_pipeline"}
    collect_summary = {"mode": "smoke", "skipped": "collect_jobs_pipeline"}
coverage_summary = build_coverage_report_pipeline()
verify_summary, collect_summary, coverage_summary

In [ ]:
staging = pd.read_csv(paths.staging_jobs_path)
display(staging.head())

print(paths.first_snapshot_path)
print(paths.coverage_report_path)
print(json.dumps(json.loads(paths.coverage_report_path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))

## 표시값, 분석용, 원문 비교 보기

`display`, `분석용`, `raw/detail` 기반 긴 텍스트를 같은 행에서 비교한다. 전처리 손실이 어느 단계에서 생겼는지 바로 확인할 수 있다.

In [ ]:
import json
import pandas as pd
from IPython.display import display

from jobs_market_v2.html_utils import clean_html_text
from jobs_market_v2.settings import get_paths

paths = get_paths()
staging = pd.read_csv(paths.staging_jobs_path)
raw_detail = pd.read_json(paths.raw_detail_path, lines=True)


def pick(payload, *keys):
    if not isinstance(payload, dict):
        return ""
    for key in keys:
        value = payload.get(key, "")
        if value:
            return value
    return ""

raw_columns = ["job_key", "주요업무_raw", "자격요건_raw", "우대사항_raw", "핵심기술_raw", "상세본문_raw"]
if raw_detail.empty or "raw_payload_json" not in raw_detail.columns:
    compare_df = pd.DataFrame(columns=raw_columns)
else:
    raw_detail["raw_payload"] = raw_detail["raw_payload_json"].apply(json.loads)
    compare_df = raw_detail.copy()
    compare_df["주요업무_raw"] = compare_df["raw_payload"].apply(lambda x: pick(x, "responsibilities", "main_tasks", "description_html", "description"))
    compare_df["자격요건_raw"] = compare_df["raw_payload"].apply(lambda x: pick(x, "requirements", "description_html", "description"))
    compare_df["우대사항_raw"] = compare_df["raw_payload"].apply(lambda x: pick(x, "preferred", "description_html", "description"))
    compare_df["핵심기술_raw"] = compare_df["raw_payload"].apply(lambda x: pick(x, "core_skills", "skills", "description_html", "description"))
    compare_df["상세본문_raw"] = compare_df["raw_payload"].apply(lambda x: pick(x, "description_html", "description"))

    for col in ["주요업무_raw", "자격요건_raw", "우대사항_raw", "핵심기술_raw", "상세본문_raw"]:
        compare_df[col] = compare_df[col].fillna("").apply(clean_html_text)

comparison_view = staging.merge(
    compare_df[["job_key", "주요업무_raw", "자격요건_raw", "우대사항_raw", "핵심기술_raw", "상세본문_raw"]],
    on="job_key",
    how="left",
)

comparison_view = comparison_view[[
    "회사명_표시",
    "공고제목_표시",
    "직무명_표시",
    "주요업무_표시",
    "주요업무_분석용",
    "주요업무_raw",
    "자격요건_표시",
    "자격요건_분석용",
    "자격요건_raw",
    "우대사항_표시",
    "우대사항_분석용",
    "우대사항_raw",
    "핵심기술_표시",
    "핵심기술_분석용",
    "핵심기술_raw",
    "상세본문_분석용",
    "상세본문_raw",
]]

display(comparison_view.head(20))


In [ ]:
# 특정 공고 1건을 세로로 길게 비교
if comparison_view.empty:
    print("comparison_view is empty")
else:
    target_offset = min(4, len(comparison_view))
    row = comparison_view.iloc[-target_offset]

    for col in comparison_view.columns:
        print(f"\n[{col}]")
        print(row[col])
